# 03: Ordenar, limitar y filtrar: controlando los resultados

ORDER BY, LIMIT, DISTINCT, LIKE, BETWEEN e IN — las herramientas para refinar tus consultas con precisión.


Ya sabes pedir datos con SELECT y filtrar con WHERE. 

En el mundo real, tus consultas necesitan más control: 

- "dame los 10 pedidos más grandes"
- "busca clientes cuyo nombre empiece por A"
- "muestra productos entre 20 y 50 euros"
- "dame las ciudades únicas". 

Hoy vas a aprender las herramientas que te dan ese control quirúrgico sobre los resultados.

## ORDER BY: ordenar resultados


```sql
-- Ordenar productos por precio (menor a mayor  = ASC, default)
SELECT nombre, precio
FROM productos
ORDER BY precio;

-- Ordenar de mayor a menor (DESC)
SELECT nombre, precio
FROM productos
ORDER BY precio DESC;

-- Ordenar por múltiples columnas (la segunda desempata)
SELECT nombre, ciudad
FROM clientes
ORDER BY ciudad ASC, nombre ASC
```

- ORDER BY precio → sin especificar, usa ASC (ascendente: 10, 20, 50, 100...). Es el default.
- ORDER BY precio DESC → descendente (100, 50, 20, 10...). DESC es explícito.
- ORDER BY ciudad ASC, nombre ASC → ordena primero por ciudad (A→Z), y dentro de la misma ciudad, por nombre (A→Z). La segunda columna DESEMPATA: sin ella, dos clientes de la misma ciudad podrían salir en cualquier orden.

**Regla práctica:** si tu ORDER BY va con LIMIT (un "Top 5"), termina siempre con una columna única como el id. 

- Sin ella, el "Top 5" de hoy puede no ser el "Top 5" de mañana — porque entre filas empatadas, SQL elige las que le vengan bien.

Sin ORDER BY, SQL NO garantiza ningún orden en los resultados — ni siquiera el orden de inserción. Si necesitas orden, pídelo siempre explícitamente.



Cuando ordenas por una columna que tiene huecos (valores que faltan), cada motor decide dónde los coloca. 

1. Aquí, con DuckDB, van al final. 
2. En PostgreSQL, cuando pides DESC los huecos van PRIMERO — la misma consulta que aquí te da los 5 pedidos más grandes, allí te devuelve pedidos sin importe. 

**La solución:** 
- ORDER BY importe DESC NULLS LAST. Funciona igual en todos los motores.
- Cógelo como costumbre: siempre que ordenes por una columna que pueda tener huecos, escribe NULLS LAST.



## LIMIT: limitar el número de resultados


```sql
-- Los 5 productos más caros
SELECT nombre, precio
FROM productos
ORDER BY precio DESC
LIMIT 5;

-- Paginación: saltar los primeros 5 y mostrar los siguientes 5
SELECT nombre, precio
FROM productos
ORDER BY precio DESC
LIMIT 5 OFFSET 5;
```

- ORDER BY precio DESC + LIMIT 5 → los 5 productos más caros. Primero ordena TODOS, luego corta en 5.
- LIMIT 5 OFFSET 5 → salta los primeros 5 resultados y muestra los siguientes 5. Es el patrón de paginación: página 1 = OFFSET 0, página 2 = OFFSET 5, etc.

**Nota:**
- OFFSET puede ser ineficiente en tablas grandes porque el motor debe calcular y descartar todas las filas anteriores. 
- Para paginación masiva, hay patrones mejores (cursor-based pagination).



**LIMIT sin ORDER BY es una lotería:**
- obtienes "5 filas cualquiera" sin garantía de cuáles. 
- SIEMPRE combina LIMIT con ORDER BY para resultados predecibles.



## DISTINCT: eliminar duplicados

Cuando exploras una tabla nueva, una de las primeras preguntas es 
- «¿qué valores distintos hay aquí?». 
- No cuántas filas: cuántos valores diferentes. 

Los 20.000 pedidos de esta base de datos tienen un estado cada uno, pero estados diferentes hay tres o cuatro. 

1. DISTINCT colapsa las repeticiones y te deja cada valor una sola vez. 
2. Es la forma más rápida de saber qué contiene una columna antes de filtrar por ella 
3.  antes de escribir WHERE estado = 'completado', conviene mirar con DISTINCT que el valor se escribe así y no "Completado" ni "COMPLETADO".



**Ojo con una cosa:** 

1. DISTINCT no se aplica a una columna, se aplica a la fila entera que hayas pedido. 
2. Si escribes SELECT DISTINCT estado, ciudad, no te da los estados distintos: te da las combinaciones distintas de estado y ciudad, que son muchas más.

```sql
-- En qué ciudades tenemos clientes?
SELECT DISTINCT ciudad
FROM clientes
ORDER BY ciudad;

-- Qué estados de pedido existen?
SELECT DISTINCT estado
FROM pedidos;
```

- SELECT DISTINCT ciudad FROM clientes → de los más de 5.000 clientes salen 15 filas: las 14 ciudades más una vacía, la de los clientes de los que no consta.
- SELECT DISTINCT estado FROM pedidos → muestra los valores posibles de la columna estado (completado, cancelado, devuelto).

DISTINCT actúa sobre TODAS las columnas del SELECT. 

Si pones SELECT DISTINCT ciudad, nombre, elimina filas donde ciudad+nombre sean idénticos (no solo ciudad).



**Nota:**

Un detalle que verás en cuanto ejecutes el primer ejemplo: entre las ciudades te va a salir una fila vacía:

1. Es correcto — hay 375 clientes de los que no consta la ciudad, y «no consta» también es un valor distinto. 
2. DISTINCT no lo esconde

Hay una lección más adelante dedicada entera a los datos que faltan.

## LIKE, BETWEEN e IN: filtros avanzados

1. **LIKE:** busca patrones dentro de un texto. 
    - El símbolo % significa «aquí puede haber cualquier cosa, o nada», y el
    - _ significa «aquí hay exactamente un carácter». 
    - Así, 'A%' es «empieza por A», '%SQL%' es «contiene SQL en algún sitio» y '_a%' es «la segunda letra es una a».

2. **BETWEEN:** es un rango, y los dos extremos entran. 
    - precio BETWEEN 25 AND 80 incluye los productos que cuestan exactamente 25 y los que cuestan exactamente 80. 
    - Es lo mismo que escribir precio >= 25 AND precio <= 80, pero se lee mejor. 
    - Para fechas tiene una trampa que ya viste en la lección anterior; para números es seguro.

3. **IN:** es una lista de valores válidos para la misma columna. 
    - estado IN ('cancelado', 'devuelto') es lo mismo que estado = 'cancelado' OR estado = 'devuelto'
    - A partir de tres o cuatro valores la diferencia de legibilidad es enorme.



```sql
-- LIKE: patrones de texto (% = cualquier cosa | _ = 1 carácter)
SELECT nombre
FROM clientes
WHERE nombre LIKE 'A%';

-- BETWEEN: rangos inclusivos
SELECT nombre, precio
FROM productos
WHERE precio BETWEEN 30 AND 100;

-- IN: lista de valores (reemplaza múltiples OR)
SELECT nombere, ciudad
FROM clientes
WHERE ciudad IN ('Madrid', 'Barcelona', 'Valencia');
```

Tres operadores de filtrado avanzado:

- LIKE 'A%' → patrón de texto. % = "cualquier cantidad de caracteres". Encuentra nombres que empiezan por A (Ana, Alberto, Alejandra...).
- BETWEEN 30 AND 100 → rango inclusivo en ambos extremos. Equivale a >= 30 AND <= 100 pero más legible.
- IN ('Madrid', 'Barcelona', 'Valencia') → lista de valores aceptados. Equivale a OR encadenados pero mucho más limpio.

Estos tres operadores se usan dentro de WHERE como alternativas más expresivas a los operadores básicos (=, >, <).

**Consejo de senior:** 

La primera vez que hice un WHERE nombre LIKE 'ana%' y no me devolvió resultados, pensé que la tabla estaba vacía. 

- Resulta que LIKE distingue mayúsculas — y la clienta se llamaba "Ana", con A mayúscula.
-  En DuckDB y PostgreSQL tienes ILIKE (la i es de "insensitive"): WHERE nombre ILIKE 'ana%' encuentra "Ana", "ANA" y "ana". 
-  Si tu motor no tiene ILIKE, el truco universal es LOWER(): WHERE LOWER(nombre) LIKE 'ana%'



**Regla mnemotécnica para el orden de las cláusulas: SELECT → FROM → WHERE → ORDER BY → LIMIT.**

El orden de escritura es fijo en SQL — no puedes poner LIMIT antes de WHERE.



## Ordenar texto no es ordenar números


Fíjate en algo de los datos que tienes aquí. 

Hay tres cafeteras del mismo modelo, de 2, 6 y 12 tazas. 

- Ordénalas por nombre y mira el resultado: 
    1. primero sale «Cafetera Espresso 12»,
    2. Después la de 2 
    3. Al final la de 6. 
- No es un fallo del motor. El nombre es texto, y el texto se ordena carácter a carácter: compara el «1» del 12 con el «2», el 1 va antes
- El tamaño del número no se mira nunca. 

Es la razón por la que un listado de versiones ordenado alfabéticamente pone la 10 antes de la 9, y por la que un informe ordenado por un código de producto que lleva números dentro sale en un orden que nadie esperaba.

